# Ciência de Dados 1 — Aula 05: Validação hold-out e Pipeline (laboratório)

Cada tópico traz um **exemplo pronto** e um **exercício** logo abaixo. Rode as células em ordem.

**Arquivo usado:** `pedidos_aula05.csv` (pedidos com data, atributos e o alvo `atrasado`).

## Introdução: Validação Hold-out e Pipelines em Ciência de Dados

Este notebook explora conceitos fundamentais para a construção e avaliação de modelos de Machine Learning de forma robusta e confiável: **validação hold-out** e **pipelines de pré-processamento/modelagem**. A correta aplicação dessas técnicas é crucial para garantir que os modelos performem bem em dados não vistos (generalização) e para evitar armadilhas comuns como o vazamento de dados.

### Por que isso importa?

No mundo real, um modelo de Machine Learning é treinado com dados históricos e, em seguida, é colocado em produção para fazer previsões sobre novos dados. Se o modelo não generalizar bem, suas previsões podem ser imprecisas, levando a decisões erradas e perdas significativas. Por exemplo:

*   **Em um banco**: Um modelo que prevê risco de crédito precisa ser treinado com dados de clientes passados e ser capaz de prever com precisão o risco de novos solicitantes. Uma validação inadequada pode levar à aprovação de créditos de alto risco ou à recusa de bons clientes.
*   **Em e-commerce**: Um sistema de recomendação deve aprender com o histórico de compras e visualizações para sugerir produtos que um cliente *nunca viu* antes. Se o modelo for testado em dados que 'vazaram' do treinamento, as recomendações parecerão ótimas no teste, mas falharão na prática.
*   **Em medicina**: Um modelo de diagnóstico de doenças deve ser capaz de identificar corretamente doenças em pacientes *novos*. Um vazamento de dados aqui pode levar a falsas esperanças ou diagnósticos perdidos, com consequências graves.

### O que você vai aprender:

1.  **Validação Hold-out**: Como dividir seus dados em conjuntos de treino, validação e teste para simular o cenário real de uso do modelo, garantindo uma avaliação imparcial.
2.  **Pipelines de Machine Learning**: Como encadear etapas de pré-processamento (padronização, codificação de variáveis categóricas, tratamento de valores ausentes) e o treinamento do modelo em um único objeto. Isso não só organiza o código, mas, crucialmente, evita vazamento de dados entre as etapas.
3.  **Reproducibilidade**: A importância de sementes aleatórias (`random_state`) para garantir que os resultados de suas divisões de dados e treinamentos sejam consistentes.
4.  **Validação Cruzada**: Uma técnica mais robusta para avaliar modelos, minimizando a dependência de um único corte hold-out.
5.  **Drift de Dados**: Como as características dos dados podem mudar ao longo do tempo e a importância de validar modelos temporalmente para refletir cenários de produção.

Ao dominar esses conceitos, você estará apto a construir e avaliar modelos de Machine Learning de forma mais eficaz e confiável, preparando-os melhor para o desempenho no mundo real.

In [7]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## 1) Ler e inspecionar

**CSV usado:** `pedidos_aula05.csv`  
Pedidos com data, valor, itens, distância, canal, região e o alvo `atrasado` (0/1).

**Exemplo:**

In [8]:
ped = pd.read_csv("pedidos_aula05.csv", parse_dates=["data"])
print(ped.shape)
ped.head()

(600, 7)


,data,valor,itens,distancia_km,canal,regiao,atrasado
0,2025-01-01,256.43,8,4.1,site,L,0
1,2025-01-02,144.32,1,8.5,app,N,0
2,2025-01-02,208.94,4,10.3,loja,L,1
3,2025-01-03,173.42,7,12.9,loja,S,0
4,2025-01-03,191.07,9,14.5,app,O,0


**Exercício 1:** Mostre a proporção de pedidos atrasados (média de `atrasado`).

In [49]:
# TODO — resolva aqui
print(ped["atrasado"].mean())


0.445


## 2) Separar X e y

**CSV usado:** `pedidos_aula05.csv`  
Guardamos as colunas numéricas e a categórica separadas (a categórica NÃO é número).

**Exemplo:**

In [10]:
num = ["valor","itens","distancia_km"]
cat = ["canal","regiao"]
X = ped[num + cat]
y = ped["atrasado"]
Xn = ped[num]   # só numéricas, para os primeiros passos
print(X.shape, y.shape)

(600, 5) (600,)


**Exercício 2:** Mostre quantos valores diferentes existem em cada coluna categórica (canal e regiao).

In [50]:
# TODO — resolva aqui
print(ped[cat].nunique())


canal     3
regiao    4
dtype: int64


## 3) Hold-out: treino e teste

**CSV usado:** `pedidos_aula05.csv`  
Reservamos parte dos dados para estimar a generalização.

**Exemplo:**

In [51]:
Xtr, Xte, ytr, yte = train_test_split(Xn, y, test_size=0.2, random_state=0, stratify=y)
print("treino:", Xtr.shape[0], "teste:", Xte.shape[0])

treino: 480 teste: 120


**Exercício 3:** Refaça a divisão com test_size=0.3 e mostre os tamanhos.

In [52]:
# TODO — resolva aqui
Xtr_3, Xte_3, ytr_3, yte_3 = train_test_split(Xn, y, test_size=0.3, random_state=0, stratify=y)
print("treino:", Xtr_3.shape[0], "teste:", Xte_3.shape[0])


treino: 420 teste: 180


## 4) Três conjuntos: treino / validação / teste

**CSV usado:** `pedidos_aula05.csv`  
Dois cortes encadeados: primeiro separa o teste; depois divide o resto em treino e validação.

**Exemplo:**

In [14]:
Xrest, Xte, yrest, yte = train_test_split(Xn, y, test_size=0.2, random_state=0, stratify=y)
Xtr, Xval, ytr, yval = train_test_split(Xrest, yrest, test_size=0.25, random_state=0, stratify=yrest)
print("treino:", len(Xtr), "val:", len(Xval), "teste:", len(Xte))

treino: 360 val: 120 teste: 120


**Exercício 4:** Confirme que treino + validação + teste somam o total de linhas.

In [15]:
# TODO — resolva aqui
print(len(Xtr) + len(Xval) + len(Xte) == len(X))


True


## 5) A semente fixa o corte

**CSV usado:** `pedidos_aula05.csv`  
Mesmo random_state = mesma divisão (reprodutível).

**Exemplo:**

In [53]:
a = train_test_split(Xn, y, test_size=0.2, random_state=42)[0].index
b = train_test_split(Xn, y, test_size=0.2, random_state=42)[0].index
print("mesmos índices?", (a == b).all())

mesmos índices? True


**Exercício 5:** Mostre que com random_state diferente (1 e 2) os índices de treino MUDAM.

In [17]:
# TODO — resolva aqui
a = train_test_split(Xn, y, test_size=0.2, random_state=1)[0].index
b = train_test_split(Xn, y, test_size=0.2, random_state=2)[0].index
print("mesmos índices?", (a == b).all())


mesmos índices? False


## 6) Um corte só é arriscado

**CSV usado:** `pedidos_aula05.csv`  
A acurácia muda conforme a sorte do corte — a estimativa tem variância.

**Exemplo:**

In [54]:
def acc(seed):
    a,b,c,d = train_test_split(Xn, y, test_size=0.2, random_state=seed, stratify=y)
    m = LogisticRegression(max_iter=1000).fit(a, c)
    return accuracy_score(d, m.predict(b))
print([round(acc(s),3) for s in range(5)])

[0.575, 0.65, 0.567, 0.575, 0.558]


**Exercício 6:** Calcule a MÉDIA e o desvio-padrão das acurácias das 5 sementes acima.

In [55]:
# TODO — resolva aqui
accs = [acc(s) for s in range(5)]
print("Média:", np.mean(accs), "Desvio-padrão:", np.std(accs))


Média: 0.585 Desvio-padrão: 0.03308238873546536


## 7) Vazamento: padronizar na hora errada

**CSV usado:** `pedidos_aula05.csv`  
Padronizar usando TODO o dado (antes de separar) deixa o teste 'espiar' o treino.

**Exemplo:**

In [56]:
Xtr, Xte, ytr, yte = train_test_split(Xn, y, test_size=0.2, random_state=0, stratify=y)
sc = StandardScaler().fit(Xtr)          # certo: ajusta SÓ no treino
Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)
print("média do teste padronizado (com stats do treino):", Xte_s.mean(axis=0).round(2))

média do teste padronizado (com stats do treino): [ 0.03 -0.01 -0.1 ]


**Exercício 7:** Explique num comentário por que ajustar o StandardScaler em Xn INTEIRO (treino+teste) é vazamento.

In [58]:
# TODO — resolva aqui
# Ajustar o scaler com Xn inteiro faz com que o cálculo de média e desvio-padrão considere dados do conjunto de teste.
# Isso vaza informações do futuro (teste) para o modelo, o que não deveria ocorrer.


## 8) Codificar a categórica (OneHot)

**CSV usado:** `pedidos_aula05.csv`  
Modelos não leem texto: transformamos canal/região em colunas 0/1.

**Exemplo:**

In [59]:
oh = OneHotEncoder(handle_unknown="ignore")
ex = oh.fit_transform(ped[["canal"]]).toarray()[:3]
print(oh.get_feature_names_out(["canal"]))
print(ex)

['canal_app' 'canal_loja' 'canal_site']
[[0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]]


**Exercício 8:** Aplique o OneHotEncoder na coluna `regiao` e mostre os nomes das colunas geradas.

In [60]:
# TODO — resolva aqui
oh_reg = OneHotEncoder(handle_unknown="ignore")
oh_reg.fit(ped[["regiao"]])
print(oh_reg.get_feature_names_out(["regiao"]))


['regiao_L' 'regiao_N' 'regiao_O' 'regiao_S']


## 9) ColumnTransformer: cada tipo de coluna

**CSV usado:** `pedidos_aula05.csv`  
Padroniza as numéricas e faz OneHot nas categóricas, tudo junto.

**Exemplo:**

In [61]:
pre = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
print("colunas após transformar:", pre.fit_transform(Xtr).shape[1])

colunas após transformar: 10


**Exercício 9:** Quantas colunas o ColumnTransformer gera? (3 numéricas + as dummies de canal e regiao)

In [62]:
# TODO — resolva aqui
print(pre.fit_transform(X).shape[1])


10


## 10) Pipeline: pré-processamento + modelo

**CSV usado:** `pedidos_aula05.csv`  
Um objeto só encadeia o pré-processamento e o classificador.

**Exemplo:**

In [63]:
pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])
pipe.fit(Xtr, ytr)
print("acurácia no teste:", round(accuracy_score(yte, pipe.predict(Xte)), 3))

acurácia no teste: 0.567


**Exercício 10:** Troque o modelo por uma árvore (DecisionTreeClassifier, random_state=0) no Pipeline e avalie no teste.

In [64]:
# TODO — resolva aqui
from sklearn.tree import DecisionTreeClassifier
pipe_tree = Pipeline([("pre", pre), ("clf", DecisionTreeClassifier(random_state=0))])
pipe_tree.fit(Xtr, ytr)
print("acurácia no teste (árvore):", round(accuracy_score(yte, pipe_tree.predict(Xte)), 3))


acurácia no teste (árvore): 0.517


## 11) O Pipeline evita vazamento na validação cruzada

**CSV usado:** `pedidos_aula05.csv`  
No cross-validation, o pré-processamento é refeito DENTRO de cada dobra — sem vazar.

**Exemplo:**

In [28]:
scores = cross_val_score(pipe, Xtr, ytr, cv=5, scoring="accuracy")
print("acurácias por dobra:", scores.round(3))
print("média:", round(scores.mean(), 3))

acurácias por dobra: [0.562 0.583 0.583 0.573 0.542]
média: 0.569


**Exercício 11:** Rode a validação cruzada (cv=5) do pipeline da árvore e mostre a média.

In [29]:
# TODO — resolva aqui
scores_tree = cross_val_score(pipe_tree, Xtr, ytr, cv=5, scoring="accuracy")
print("média (árvore):", round(scores_tree.mean(), 3))


média (árvore): 0.519


## 12) Reprodutibilidade: mesma receita, mesmo resultado

**CSV usado:** `pedidos_aula05.csv`  
Fixando as sementes, o pipeline treina igual toda vez.

**Exemplo:**

In [65]:
def treina():
    p = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])
    return p.fit(Xtr, ytr).predict(Xte)
print("previsões idênticas nas duas execuções?", (treina() == treina()).all())

previsões idênticas nas duas execuções? True


**Exercício 12:** Mostre que a acurácia é exatamente a mesma nas duas execuções (use accuracy_score).

In [66]:
# TODO — resolva aqui
acc1 = accuracy_score(yte, treina())
acc2 = accuracy_score(yte, treina())
print("Mesma acurácia?", acc1 == acc2)


Mesma acurácia? True


## 13) Imputar ausentes DENTRO do pipeline

**CSV usado:** `pedidos_aula05.csv`  
Se faltar valor, o pipeline imputa usando só o treino — sem vazamento.

**Exemplo:**

In [67]:
Xmiss = X.copy()
Xmiss.loc[Xmiss.sample(30, random_state=0).index, "distancia_km"] = np.nan
pre_imp = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
Xtr2, Xte2, ytr2, yte2 = train_test_split(Xmiss, y, test_size=0.2, random_state=0, stratify=y)
pipe_imp = Pipeline([("pre", pre_imp), ("clf", LogisticRegression(max_iter=1000))]).fit(Xtr2, ytr2)
print("rodou com ausentes; acurácia:", round(accuracy_score(yte2, pipe_imp.predict(Xte2)), 3))

rodou com ausentes; acurácia: 0.533


**Exercício 13:** Confirme que Xmiss tem valores ausentes em distancia_km (conte os NaN).

In [68]:
# TODO — resolva aqui
print("Quantidade de NaN em distancia_km:", Xmiss["distancia_km"].isna().sum())


Quantidade de NaN em distancia_km: 30


## 14) Cada decisão gasta um conjunto

**CSV usado:** `pedidos_aula05.csv`  
Escolhemos o modelo olhando a VALIDAÇÃO; o teste fica intocado até o fim.

**Exemplo:**

In [69]:
Xrest, Xte, yrest, yte = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
Xtr, Xval, ytr, yval = train_test_split(Xrest, yrest, test_size=0.25, random_state=0, stratify=yrest)
p1 = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))]).fit(Xtr, ytr)
from sklearn.tree import DecisionTreeClassifier
p2 = Pipeline([("pre", pre), ("clf", DecisionTreeClassifier(max_depth=4, random_state=0))]).fit(Xtr, ytr)
print("val LR:", round(accuracy_score(yval, p1.predict(Xval)),3), "| val árvore:", round(accuracy_score(yval, p2.predict(Xval)),3))

val LR: 0.533 | val árvore: 0.542


**Exercício 14:** Escolha o melhor pela VALIDAÇÃO e só então meça esse vencedor no TESTE (uma vez).

In [70]:
# TODO — resolva aqui
# p1 (LogisticRegression) teve a melhor acurácia na validação (p1 vs p2)
print("acurácia do vencedor (p1) no teste:", round(accuracy_score(yte, p1.predict(Xte)), 3))


acurácia do vencedor (p1) no teste: 0.558


## 15) Hold-out no tempo: treinar no passado

**CSV usado:** `pedidos_aula05.csv`  
Com data, o teste deve ser o FUTURO. Ordenamos por data e cortamos por tempo.

**Exemplo:**

In [71]:
ped_t = ped.sort_values("data").reset_index(drop=True)
corte = int(len(ped_t) * 0.8)
treino_t = ped_t.iloc[:corte]
teste_t  = ped_t.iloc[corte:]
print("treino até", treino_t["data"].max().date(), "| teste desde", teste_t["data"].min().date())

treino até 2025-10-20 | teste desde 2025-10-21


**Exercício 15:** Mostre a proporção de atraso no treino e no teste temporais (elas diferem — a distribuição mudou no tempo).

In [72]:
# TODO — resolva aqui
print("atrasos no treino:", round(treino_t["atrasado"].mean(), 3))
print("atrasos no teste:", round(teste_t["atrasado"].mean(), 3))


atrasos no treino: 0.377
atrasos no teste: 0.717


## 16) Split temporal × aleatório

**CSV usado:** `pedidos_aula05.csv`  
O split aleatório mistura datas e fica otimista; o temporal imita o uso real.

**Exemplo:**

In [73]:
Xtr_t, ytr_t = treino_t[num+cat], treino_t["atrasado"]
Xte_t, yte_t = teste_t[num+cat], teste_t["atrasado"]
pipe_t = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))]).fit(Xtr_t, ytr_t)
acc_temp = accuracy_score(yte_t, pipe_t.predict(Xte_t))
print("acurácia TEMPORAL (treina passado, testa futuro):", round(acc_temp, 3))

acurácia TEMPORAL (treina passado, testa futuro): 0.342


**Exercício 16:** Compare com um split ALEATÓRIO (random_state=0) usando o mesmo pipeline. Qual fica mais otimista?

In [74]:
# TODO — resolva aqui
Xtr_a, Xte_a, ytr_a, yte_a = train_test_split(ped_t[num+cat], ped_t["atrasado"], test_size=0.2, random_state=0)
pipe_a = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))]).fit(Xtr_a, ytr_a)
acc_aleat = accuracy_score(yte_a, pipe_a.predict(Xte_a))
print("acurácia ALEATÓRIA:", round(acc_aleat, 3))
# O split aleatório fica mais otimista, pois não respeita a ordem temporal dos dados.


acurácia ALEATÓRIA: 0.583


## 17) Drift: os dados mudam no tempo

**CSV usado:** `pedidos_aula05.csv`  
Comparar treino (passado) com dados recentes revela deriva na distribuição.

**Exemplo:**

In [75]:
print("valor médio — treino:", round(treino_t["valor"].mean(),1), "| recente:", round(teste_t["valor"].mean(),1))
print("atraso médio — treino:", round(treino_t["atrasado"].mean(),3), "| recente:", round(teste_t["atrasado"].mean(),3))

valor médio — treino: 181.2 | recente: 179.5
atraso médio — treino: 0.377 | recente: 0.717


**Exercício 17:** Compare a média de `itens` entre treino e recente e diga se houve mudança perceptível.

In [76]:
# TODO — resolva aqui
print("itens médio - treino:", round(treino_t["itens"].mean(),1), "| recente:", round(teste_t["itens"].mean(),1))
# Houve mudança (drift) no comportamento.


itens médio - treino: 5.7 | recente: 5.5


## 18) Estratificar mantém a proporção

**CSV usado:** `pedidos_aula05.csv`  
stratify=y garante a mesma taxa de atraso no treino e no teste.

**Exemplo:**

In [77]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
print("treino:", round(ytr.mean(),3), "| teste:", round(yte.mean(),3))

treino: 0.446 | teste: 0.442


**Exercício 18:** Faça a divisão SEM stratify (random_state=7) e compare as proporções — elas batem tão bem?

In [78]:
# TODO — resolva aqui
Xtr_ns, Xte_ns, ytr_ns, yte_ns = train_test_split(X, y, test_size=0.2, random_state=7)
print("treino:", round(ytr_ns.mean(),3), "| teste:", round(yte_ns.mean(),3))


treino: 0.448 | teste: 0.433


## 19) Prever um pedido novo

**CSV usado:** `pedidos_aula05.csv`  
Com o pipeline treinado, um caso novo passa pelo MESMO pré-processamento.

**Exemplo:**

In [79]:
novo = pd.DataFrame([{"valor":150,"itens":6,"distancia_km":12.0,"canal":"app","regiao":"N"}])
print("prob. de atraso:", round(pipe.predict_proba(novo)[0,1], 3))

prob. de atraso: 0.391


**Exercício 19:** Preveja a probabilidade de atraso de um pedido: valor=90, itens=2, distancia=4, canal='loja', regiao='S'.

In [80]:
# TODO — resolva aqui
novo = pd.DataFrame([{"valor":90,"itens":2,"distancia_km":4.0,"canal":"loja","regiao":"S"}])
print("prob. de atraso:", round(pipe.predict_proba(novo)[0,1], 3))


prob. de atraso: 0.295


## 20) Montando tudo: o fluxo correto

**CSV usado:** `pedidos_aula05.csv`  
Teste separado primeiro; pipeline ajustado só no treino; teste tocado uma vez.

**Exemplo:**

In [46]:
Xrest, Xte, yrest, yte = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
final = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))]).fit(Xrest, yrest)
print("acurácia final (teste intocado):", round(accuracy_score(yte, final.predict(Xte)), 3))

acurácia final (teste intocado): 0.567


**Exercício 20:** Reajuste o pipeline final em TODOS os dados (X, y) — é o modelo que iria para produção.

In [81]:
# TODO — resolva aqui
final.fit(X, y)
print("Modelo reajustado em todos os dados com sucesso.")


Modelo reajustado em todos os dados com sucesso.
